# Cette exercice a ppour but de m'initié aux vrai dataset
# On ferra l'exerice classique du Titatic

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

In [2]:
df = sns.load_dataset('titanic')
print(df.head())

   survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
0         0       3    male  22.0      1      0   7.2500        S  Third   
1         1       1  female  38.0      1      0  71.2833        C  First   
2         1       3  female  26.0      0      0   7.9250        S  Third   
3         1       1  female  35.0      1      0  53.1000        S  First   
4         0       3    male  35.0      0      0   8.0500        S  Third   

     who  adult_male deck  embark_town alive  alone  
0    man        True  NaN  Southampton    no  False  
1  woman       False    C    Cherbourg   yes  False  
2  woman       False  NaN  Southampton   yes   True  
3  woman       False    C  Southampton   yes  False  
4    man        True  NaN  Southampton    no   True  


# Mission résoudre les trous dans la colonne âge
On doit d'abord retrouver la coéfficient de variation pour savoir si on doit remplacer les âges manquantes , et on va le aire par genre, homme et femme 

In [3]:
# Mais avant tous on dois d'abord vérifier si le genre (mâle ou female) sont complet et supprimé les lignes incomplette
# On va voir combien de ligne manquante ou des faute de frappe

print(f"les valeur existant : {df['sex'].unique()}")
print(f"nombres de valeur manquante : {df['sex'].isnull().sum()}")

les valeur existant : <StringArray>
['male', 'female']
Length: 2, dtype: str
nombres de valeur manquante : 0


In [4]:
# Ensuite transformé les valeur non normale en NaN pour la colonne age
# on definit la valeur normal 
age_normale = (df['age'].notna() & (df['age']>=0) & (df['age']<=120) & (~df['age'].isin([999,9999])) )
#Remplacer les valeur non normale en NaN 
df['age'] = df['age'].where(age_normale, np.nan)


Question , avant on a utilisé panda avec replae pour transformer des valeur en NaN , maintenant on utilise numpy, pourquoi ?
#df['debit_mbps'] = df['debit_mbps'].replace(9999,pd.NA).dropna()

In [5]:
# On calcul la coefficient de variation pour homme et femme pour savoir si on doit remplacer par moyenne ou par la medianne
# 1. Calcul des statistiques par groupe (Sexe)
stats = df.groupby('sex')['age'].agg(['mean', 'std'])

# 2. Calcul du Coefficient de Variation (en %)
stats['cv'] = (stats['std'] / stats['mean']) * 100

print(stats)


             mean        std         cv
sex                                    
female  27.915709  14.110146  50.545542
male    30.726645  14.678201  47.770269


On vois que la cv est au dessus de 15% donc on utilise la medianne

In [6]:
# On remplace les NaN de la colonne 'age' par la médiane du groupe 'sexe' correspondant
df['age'] = df['age'].fillna(df.groupby('sex')['age'].transform('median'))

# Vérification du nombre de NaN restants
print(df['age'].isnull().sum())


0


# Mission complèté le port d'embarquement

In [7]:
print(f"Le nombre de lieu d'embarquation manquantes avant nétoyage: {df['embarked'].isnull().sum()}")
# Le nombre de lieu d'embarquement manquant est deux, on va utiliser la mode pour les remplacer
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])
# Verification 
print(f"Le nombre de lieu d'embarquation manquantes après nétoyage: {df['embarked'].isnull().sum()}")


Le nombre de lieu d'embarquation manquantes avant nétoyage: 2
Le nombre de lieu d'embarquation manquantes après nétoyage: 0


# Mission 3:  Les Statistiques "Métier" (Business Insights)

Le prix du billet (fare) : Quelle est la moyenne et la médiane du prix du billet ? Y a-t-il des valeurs aberrantes (des gens qui ont payé beaucoup trop cher) ?

Le taux de survie : Utilise .groupby() pour savoir si on avait plus de chances de survivre en étant en 1ère classe (pclass = 1) ou en 3ème classe (pclass = 3).

In [8]:
# La moyenne du prix du billet
fare_stats =  df['fare'].agg(['mean','median','std'])
print(fare_stats)

mean      32.204208
median    14.454200
std       49.693429
Name: fare, dtype: float64


# Commencement pour l'IA
Étape 1 : Préparer la Data pour le Machine Learning
Les algorithmes d'IA ne comprennent que les chiffres. Ils ne savent pas traiter du texte comme "male" ou "female". On va donc encoder ces variables en 0 ou 1

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

#1. Sélection des colonnes utiles et propres
features = ['pclass', 'sex', 'age', 'fare']
X = df[features].copy()
y = df['survived']

# 2. Encodage du sexe : 'female' -> 0, 'male' -> 1
X['sex'] = X['sex'].map({'female': 0, 'male': 1})

print("--- Aperçu de X (les variables d'entrée) ---")
print(X.head())

--- Aperçu de X (les variables d'entrée) ---
   pclass  sex   age     fare
0       3    1  22.0   7.2500
1       1    0  38.0  71.2833
2       3    0  26.0   7.9250
3       1    0  35.0  53.1000
4       3    1  35.0   8.0500


# Étape 2 : Séparer Train / Test & Entraîner l'IA
En Machine Learning, on réserve toujours une partie des données (ex: 20%) pour tester l'IA sur des passagers qu'elle n'a jamais vus. C'est ainsi qu'on mesure sa vraie précision.

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Découpage : 80% pour apprendre, 20% pour tester
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Création et entraînement de l'algorithme (Random Forest)
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# 3. Évaluation de la précision sur les données de TEST
y_pred = model.predict(X_test)
precision = accuracy_score(y_test, y_pred)

print(f"Précision de ton modèle d'IA : {precision * 100:.2f}%")

Précision de ton modèle d'IA : 82.68%


# Étape 3 : Exporter le Model.pkl (La brique FastAPI du prof)
Maintenant que le modèle est entraîné, on va le "figer" sous forme de fichier. C'est ce fichier .pkl que ton serveur FastAPI chargera plus tard dans Docker pour répondre aux requêtes HTTP.

In [12]:
import joblib

# Sauvegarde du cerveau de l'IA dans un fichier
joblib.dump(model, 'Model.pkl')
print("Le fichier 'Model.pkl' a été généré avec succès !")

Le fichier 'Model.pkl' a été généré avec succès !


# Test d'Inférence (La simulation FastAPI)
Imagine qu'un utilisateur envoie un JSON à ton futur serveur FastAPI pour tester la survie d'un passager :

Passager : 3ème classe (pclass=3), Homme (sex=1), 22 ans (age=22), Billet à 7.25$ (fare=7.25).

In [14]:
# Charger le modèle sauvegardé (comme le fera FastAPI)
modele_charge = joblib.load('Model.pkl')

# Faire la prédiction pour le nouveau passager
""" nouveau_passager = np.array([[3, 1, 22.0, 7.25]])
prediction = modele_charge.predict(nouveau_passager)

if prediction[0] == 1:
    print("L'IA prédit : Le passager aurait SURVÉCU 🟢")
else:
    print("L'IA prédit : Le passager aurait SUCCOMBÉ 🔴") """

un_passager = pd.DataFrame([[3, 1, 22.0, 7.25]], columns=['pclass', 'sex', 'age', 'fare'])
prediction = modele_charge.predict(un_passager)

if prediction[0] == 1:
    print("L'IA prédit : Le passager aurait SURVÉCU 🟢")
else:
    print("L'IA prédit : Le passager aurait SUCCOMBÉ 🔴")

L'IA prédit : Le passager aurait SUCCOMBÉ 🔴
